In [5]:
import pandas as pd
import os

# --- 1. Load our clean V2 label list ---
base_path = '/content/drive/MyDrive/AI_Physio_Coach/dataset/'
clean_label_file = os.path.join(base_path, 'df_v2_labels_CLEAN.csv')
df_v2_labels_clean = pd.read_csv(clean_label_file)

# --- 2. Get the filename of the first sample ---
first_sample = df_v2_labels_clean.iloc[0]
base_filename = first_sample['filename'] # e.g., "G1A-CTK-R6-Roscoff-101"

# --- 3. Construct the correct Kinect filename ---
# We split the base_filename into parts, e.g., ['G1A', 'CTK', 'R6', 'Roscoff', '101']
parts = base_filename.split('-')
if len(parts) >= 2:
    # Construct the new name: G1A + -Kinect- + (rest of the name)
    kinect_filename = f"{parts[0]}-Kinect-{'-'.join(parts[1:])}.txt"

    kinect_path = os.path.join(base_path, 'kinect')
    kinect_file_path = os.path.join(kinect_path, kinect_filename)

    # --- 4. Print the start of the file ---
    if os.path.exists(kinect_file_path):
        print(f"Success! Found Kinect file: {kinect_filename}")
        print(f"\n--- Inspecting first 5 lines of the file ---")

        # Use !head to print the first 5 lines
        !head -n 5 {kinect_file_path}

    else:
        print(f"Error: Could not find the constructed file: {kinect_file_path}")
        print("The naming convention might be more complex.")
else:
    print(f"Error: The base filename '{base_filename}' is not in the expected format.")

Success! Found Kinect file: G1A-Kinect-CTK-R6-Roscoff-101.txt

--- Inspecting first 5 lines of the file ---
0.739662 -0.359348 3.410063 -0.093452 0.023524 0.990231 -0.100773 0.747254 -0.091634 3.354261 -0.092161 0.023392 0.990352 -0.100803 0.751788 0.171667 3.286486 -0.091911 0.017963 0.985617 -0.140676 0.761476 0.296226 3.238444 0.000000 0.000000 0.000000 0.000000 0.598211 0.084960 3.320846 0.103597 0.753167 -0.648104 0.044370 0.565625 -0.137633 3.361842 0.040249 0.850701 -0.108022 -0.512854 0.656687 -0.295407 3.177175 0.014314 -0.422396 -0.442357 0.791011 0.719760 -0.351364 3.203851 -0.408169 -0.588644 -0.130485 0.685471 0.902073 0.057919 3.266456 -0.016330 0.784887 0.592573 -0.180398 0.907567 -0.150625 3.286689 0.041918 0.950005 0.027396 0.308193 0.697176 -0.191381 3.136508 0.649393 -0.460887 0.020426 0.604528 0.627900 -0.205119 3.089634 0.646591 -0.455684 0.029697 0.611057 0.665768 -0.350677 3.392836 0.007739 -0.649900 0.746131 -0.144424 0.464710 -0.411230 3.181662 0.597921 -0.3609

In [7]:
import numpy as np
import pandas as pd
import os

# --- 1. SET CONSTANTS ---
BASE_PATH = '/content/drive/MyDrive/AI_Physio_Coach/dataset/'
CLEAN_LABEL_FILE = os.path.join(BASE_PATH, 'df_v2_labels_CLEAN.csv')
KINECT_DATA_PATH = os.path.join(BASE_PATH, 'kinect')
KINECT_FPS = 30
MAX_LENGTH = 500
NUM_JOINTS = 25
NUM_FEATURES = 7

# --- 2. LOAD OUR CLEAN "ANSWER KEY" ---
print(f"Loading clean labels from {CLEAN_LABEL_FILE}...")
df_labels = pd.read_csv(
    CLEAN_LABEL_FILE,
    dtype={'filename': str}  # <-- 1. THE FIRST FIX: Force 'filename' to be a string
)

# --- 3. DEFINE THE KINECT PREPROCESSING FUNCTION ---
# (This function is correct, no changes needed)
def normalize_kinect_skeleton(skeleton_data_slice):
    normalized_data = skeleton_data_slice.copy()
    root_joint_per_frame = normalized_data[:, 0:1, :]
    centered_data = normalized_data - root_joint_per_frame
    neck_joint_position = centered_data[0, 2, :3]
    torso_length = np.linalg.norm(neck_joint_position)
    if torso_length == 0:
        return None
    scaled_data = centered_data.copy()
    scaled_data[:, :, :3] = scaled_data[:, :, :3] / torso_length
    return scaled_data

# --- 4. MAIN PROCESSING LOOP ---
print("Starting main processing loop for 503 labels...")
all_sequences = []
all_labels = []
skipped_count = 0

for index, row in df_labels.iterrows():
    base_filename = row['filename']
    start_time = row['start_time']
    end_time = row['end_time']
    label = row['label']

    # <-- 2. THE SECOND FIX: Add a safety check for bad/missing filenames
    if pd.isna(base_filename) or not isinstance(base_filename, str):
        print(f"Warning: Skipping row {index} with invalid filename: {base_filename}")
        skipped_count += 1
        continue

    # 4a. Construct the correct Kinect filename
    parts = base_filename.split('-')
    kinect_filename = f"{parts[0]}-Kinect-{'-'.join(parts[1:])}.txt"
    kinect_file_path = os.path.join(KINECT_DATA_PATH, kinect_filename)

    try:
        # 4b. Load the *entire* raw kinect data file
        raw_data = np.loadtxt(kinect_file_path)

        # 4c. Calculate frame slicing
        start_frame = int(start_time * KINECT_FPS)
        end_frame = int(end_time * KINECT_FPS)

        if end_frame <= start_frame:
            end_frame = start_frame + 1
        if end_frame > len(raw_data):
            end_frame = len(raw_data)

        # 4d. Slice the repetition from the full file
        repetition_slice = raw_data[start_frame:end_frame]

        # 4e. Reshape the slice
        num_frames = repetition_slice.shape[0]
        if num_frames == 0:
            skipped_count += 1
            continue

        structured_slice = repetition_slice.reshape(num_frames, NUM_JOINTS, NUM_FEATURES)

        # 4f. Normalize the slice
        normalized_slice = normalize_kinect_skeleton(structured_slice)

        if normalized_slice is None:
            skipped_count += 1
            continue

        # 4g. Add the processed data to our lists
        all_sequences.append(normalized_slice.astype(np.float32))
        all_labels.append(label)

    except FileNotFoundError:
        print(f"Warning: File not found, skipping. {kinect_filename}")
        skipped_count += 1
    except Exception as e:
        print(f"Warning: Error processing {kinect_filename}: {e}")
        skipped_count += 1

print(f"\nLoop complete. Successfully processed {len(all_sequences)} sequences.")
print(f"Total skipped files/errors: {skipped_count}")

# --- 5. PAD ALL SEQUENCES ---
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(f"Padding all {len(all_sequences)} sequences to max length {MAX_LENGTH}...")
padded_sequences = pad_sequences(
    all_sequences,
    maxlen=MAX_LENGTH,
    padding='post',
    dtype='float32'
)

# --- 6. SAVE THE FINAL NPZ FILE ---
labels_df = pd.DataFrame(all_labels, columns=['label'])
unique_labels = labels_df['label'].unique()
label_to_int_mapping = {label: i for i, label in enumerate(unique_labels)}
int_labels = labels_df['label'].map(label_to_int_mapping).to_numpy()

print("\n--- Label Mapping Created ---")
print(label_to_int_mapping)

v2_kinect_save_path = os.path.join(BASE_PATH, 'v2_kinect_dataset.npz')
np.savez_compressed(
    v2_kinect_save_path,
    sequences=padded_sequences,
    labels=int_labels
)

print(f"\n--- 🚀 V2 KINECT DATASET CREATION COMPLETE! ---")
print(f"Final sequences shape: {padded_sequences.shape}")
print(f"Final labels shape: {int_labels.shape}")
print(f"Dataset saved to: {v2_kinect_save_path}")

Loading clean labels from /content/drive/MyDrive/AI_Physio_Coach/dataset/df_v2_labels_CLEAN.csv...
Starting main processing loop for 503 labels...

Loop complete. Successfully processed 501 sequences.
Total skipped files/errors: 2
Padding all 501 sequences to max length 500...

--- Label Mapping Created ---
{'Correct': 0, 'Incomplete': 1, 'Motionless': 2, 'Error3': 3, 'Error2': 4, 'Other': 5, 'Error1': 6}

--- 🚀 V2 KINECT DATASET CREATION COMPLETE! ---
Final sequences shape: (501, 500, 25, 7)
Final labels shape: (501,)
Dataset saved to: /content/drive/MyDrive/AI_Physio_Coach/dataset/v2_kinect_dataset.npz


In [8]:
import os
import pandas as pd

# --- 1. Load our clean V2 label list ---
base_path = '/content/drive/MyDrive/AI_Physio_Coach/dataset/'
clean_label_file = os.path.join(base_path, 'df_v2_labels_CLEAN.csv')
df_v2_labels_clean = pd.read_csv(clean_label_file, dtype={'filename': str})

# --- 2. Get the filename of the first sample ---
first_sample = df_v2_labels_clean.iloc[0]
base_filename = first_sample['filename'] # e.g., "G1A-CTK-R6-Roscoff-101"

# --- 3. Find the corresponding OpenPose file ---
openpose_path = os.path.join(base_path, 'openpose')
# The documentation said 'dictionary', so let's guess a .json extension
openpose_filename = f"{base_filename}.json" # Guess 1
openpose_file_path = os.path.join(openpose_path, openpose_filename)

# If Guess 1 doesn't exist, try another common format
if not os.path.exists(openpose_file_path):
    # Let's try the same naming as Kinect
    parts = base_filename.split('-')
    openpose_filename = f"{parts[0]}-OpenPose-{'-'.join(parts[1:])}.json" # Guess 2
    openpose_file_path = os.path.join(openpose_path, openpose_filename)

# If Guess 2 doesn't exist, try a .txt
if not os.path.exists(openpose_file_path):
    openpose_filename = f"{base_filename}.txt" # Guess 3
    openpose_file_path = os.path.join(openpose_path, openpose_filename)


# --- 4. Print the start of the file ---
if os.path.exists(openpose_file_path):
    print(f"Success! Found OpenPose file: {openpose_filename}")
    print(f"\n--- Inspecting first 10 lines of the file ---")

    # Use !head to print the first 10 lines
    !head -n 10 {openpose_file_path}

else:
    print(f"Error: Could not find a matching OpenPose file for {base_filename}")
    print(f"Looked in: {openpose_path}")

Error: Could not find a matching OpenPose file for G1A-CTK-R6-Roscoff-101
Looked in: /content/drive/MyDrive/AI_Physio_Coach/dataset/openpose
